# Reproducing the article

Huang K et al. Scale-dependent applicability of directed mean-field percolation in soccer passing networks. *Chaos, Solitons & Fractals* 213 (2026) 119356. https://doi.org/10.1016/j.chaos.2026.119356

This notebook runs every script of the repository in the order of the article. Each script compares its output with the published values and stops with an error if any value differs. Figures are written to `figures/` and tables to `results/`.

Run it from the repository root. Steps that need the raw StatsBomb event data run only when the environment variable `DATA_ROOT` points to them; all other steps use the shipped files in `data/`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

HAVE_RAW = Path(os.environ.get("DATA_ROOT", "/path/to/statsbomb/datos")).exists()

def run(script, *args, tail=12):
    """Run one script from the repository root and show the end of its output."""
    p = subprocess.run([sys.executable, script, *args], capture_output=True, text=True)
    lines = p.stdout.strip().splitlines()
    print(f"$ python {script} {' '.join(args)}".rstrip())
    print("\n".join(lines[-tail:]))
    if p.returncode != 0:
        print(p.stderr[-2000:])
        raise RuntimeError(f"{script} exited with status {p.returncode}")
    print()

print("Raw event data available:", HAVE_RAW)

## 1. One network from raw events to thresholds

Builds the four networks of Manchester City against Burnley (2023/24, Matchday 1, Fig. 2), runs the percolation protocol of the article at L2 and compares the result with the shipped values. Needs `DATA_ROOT`.

In [ ]:
if not HAVE_RAW:
    print("DATA_ROOT not set, skipping")
else:
    import os, sys
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import networkx as nx

    DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/statsbomb/datos"))
    sys.path.insert(0, str(Path("code/pipeline").resolve()))
    from data_loader import safe_load_match_events, extract_passes, build_substitution_map, apply_substitution_map
    from network_builder import ZonalNetworkBuilder, SCALE_CONFIGS
    from robustness import (multi_trial_edge_removal, edge_removal_simulation, find_critical_threshold,
                            calculate_robustness_integral, calculate_molloy_reed_threshold)
    from compute_directed_indicators import compute_directed_indicators

    ms = pd.read_csv("data/match_summary_mc100_guarded.csv")

    # Manchester City vs Burnley, EPL 2023/24 Matchday 1 (the match of Fig. 2)
    MATCH_ID, TEAM_NAME = 3892755, "Manchester City"
    events = safe_load_match_events(str(DATA_ROOT / "2_281_2023_24" / f"{MATCH_ID}_events.json"))
    team_id = next(ev["team"]["id"] for ev in events
                   if ev.get("type", {}).get("id") == 35 and ev["team"]["name"] == TEAM_NAME)

    # completed open-play passes, substitutes inherit the node identity of the player they replace
    passes = extract_passes(events, match_id=MATCH_ID, team_id=team_id)
    passes = apply_substitution_map(passes, build_substitution_map(events, team_id=team_id))
    print(f"{len(passes)} completed open-play passes")

    builder = ZonalNetworkBuilder()
    print(f"{'Scale':5s} {'Zones':>6s} {'|V|':>5s} {'|E|':>5s}")
    networks = {}
    for scale in ["L0", "L1", "L2", "L3"]:
        G = builder.build_network(passes, scale=scale)
        networks[scale] = G
        nx_g, ny_g = SCALE_CONFIGS[scale]
        print(f"{scale:5s} {nx_g*ny_g:6d} {G.number_of_nodes():5d} {nx.DiGraph(G).number_of_edges():5d}")

    # percolation at L2 with the protocol of the article
    G = networks["L2"]
    indicators = compute_directed_indicators(G)
    sim_error = multi_trial_edge_removal(G, method="error", n_steps=50, n_trials=100, seed=42)
    sim_attack = edge_removal_simulation(G, method="attack", n_steps=50)
    fc_error, _ = find_critical_threshold(sim_error, "strong", min_pinf_fraction=0.05)
    fc_attack, _ = find_critical_threshold(sim_attack, "strong", min_pinf_fraction=0.05)
    fc_dmr = calculate_molloy_reed_threshold(G)

    row = ms[(ms.MatchID == MATCH_ID) & (ms.Team == TEAM_NAME) & (ms.Scale == "L2")].iloc[0]
    print(f"\n{'L2 quantity':22s} {'computed':>10s} {'shipped':>10s}")
    for label, value, col in [("f_c Error", fc_error, "pc_Error_Strong"),
                              ("f_c Attack", fc_attack, "pc_Attack_Strong"),
                              ("f_c dMR (Eq. 1)", fc_dmr, "pc_Theory"),
                              ("R Error", calculate_robustness_integral(sim_error, "strong"), "R_Error_Strong"),
                              ("R Attack", calculate_robustness_integral(sim_attack, "strong"), "R_Attack_Strong"),
                              ("flow hierarchy", indicators["FlowHierarchy"], "FlowHierarchy"),
                              ("mu11 / mu", indicators["Mu11_Mu"], "Mu11_Mu")]:
        print(f"{label:22s} {value:10.4f} {row[col]:10.4f}")
        assert abs(value - row[col]) < 1e-9, label
    print("All values equal the shipped results.")

## 2. Main text

In [ ]:
run("code/results_numbers_table2.py", tail=3)   # Sections 3.1 to 3.5 and Table 2
run("code/fig3_error_vs_attack.py", tail=1)
run("code/fig4_structural_factors.py", tail=1)
run("code/fig5_partial_correlation.py", tail=1)
run("code/fig6_table3_mr_benchmark.py")         # Fig. 6 and Table 3
run("code/fig7_rai_scatter.py", tail=2)
run("code/fig8_team_stability.py", tail=2)
run("code/table4_market_value.py", tail=8)      # Table 4
run("code/in_out_degree_correlation.py", tail=6)
if HAVE_RAW:
    run("code/fig2_multiscale.py", tail=3)
    run("code/fig9_baena_case.py", tail=3)

The synthetic reference networks of Section 3.4 are regenerated and compared with the shipped results by the call below (about three minutes on eight cores for all 400 realisations; `--test` runs 5 per model).

In [ ]:
run("code/synthetic_baselines_generator.py", "--test", tail=3)

## 3. Supplementary Material

In [ ]:
S = "code/supplementary/"
run(S + "s01_descriptive.py", tail=2)           # Table S1
run(S + "s02_corr_heatmap.py", tail=2)          # Fig. S1
run(S + "s03_position_mediation.py", tail=2)    # Tables S2 and S3, Fig. S2
run(S + "s04_substitute_inheritance.py", tail=2) # Section S4
run(S + "s05_threshold_sensitivity.py", tail=2) # Tables S4 and S5
run(S + "s06_scgc_vs_wgcc.py", tail=2)          # Table S6
run(S + "s07_convergence.py", tail=2)           # Fig. S3
run(S + "s08_calibration.py", tail=2)           # Table S7, about two minutes
run(S + "s08_threshold_settings.py", tail=2)    # Table S8
run(S + "s09_within_scale.py", tail=2)          # Fig. S5
run(S + "s10_rewiring_null.py", "--plot", tail=2)  # Fig. S6
run(S + "s11_degree_distributions.py", tail=2)  # Fig. S7
run(S + "s12_pca.py", tail=2)                   # Fig. S8
run(S + "s13_fc_r_integral.py", tail=2)         # Fig. S9
run(S + "s14_ebc_ranking.py", "--plot", tail=2) # Fig. S10
if HAVE_RAW:
    run(S + "s08_example_curves.py", tail=3)          # Fig. S4